In [2]:
import pandas as pd
import torch
from PIL import Image
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
from pathlib import Path
from tqdm import tqdm
import os
import re

# --- 1. 설정 (Configuration) ---
print("1. 설정을 초기화합니다...")

# [사용자 설정] 파일 경로들을 지정합니다.
DEV_TEST_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv"
SAMPLE_SUBMISSION_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv"
BASE_IMAGE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images"
FINAL_SUBMISSION_PATH = "instructblip_submission.csv"

# --- ★★★ 모델 ID 수정 ★★★ ---
MODEL_NAME = "Salesforce/instructblip-flan-t5-xl"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {DEVICE}")

# --- 2. InstructBLIP 모델 및 프로세서 로드 ---
print(f"\n2. InstructBLIP 모델({MODEL_NAME})을 로드합니다...")

# InstructBlip 전용 프로세서와 모델을 로드합니다.
processor = InstructBlipProcessor.from_pretrained(MODEL_NAME)
model = InstructBlipForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16, # GPU 메모리 절약 및 속도 향상을 위해 float16 사용
    device_map="auto"
).eval()
print("모델 로드 완료.")

# --- 3. 데이터 로드 ---
print("\n3. 데이터 로드를 시작합니다...")
try:
    df_test = pd.read_csv(DEV_TEST_CSV_PATH)
    df_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
except FileNotFoundError as e:
    print(f"오류: 필요한 파일을 찾을 수 없습니다: {e}")
    exit()

# --- 4. 추론 루프 (InstructBLIP 방식) ---
print(f"\n4. 총 {len(df_test)}개의 데이터에 대한 추론을 시작합니다...")

for index, row in tqdm(df_test.iterrows(), total=df_test.shape[0], desc="InstructBLIP 추론 진행률"):
    test_id = row['ID']
    relative_img_path = row['img_path']
    question = row['Question']
    choices = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D']}
    
    image_filename = os.path.basename(relative_img_path)
    full_image_path = Path(BASE_IMAGE_DIR) / image_filename

    try:
        image = Image.open(full_image_path).convert("RGB")
        
        # --- 핵심: 4지선다형 답변을 유도하는 프롬프트 구성 ---
        # InstructBLIP은 질문만으로도 잘 작동하지만, 명시적인 지시를 추가하면 더 안정적입니다.
        prompt = f"Question: {question} Options: A. {choices['A']}, B. {choices['B']}, C. {choices['C']}, D. {choices['D']}. Based on the image, the single best answer is:"

        # 모델 입력 준비
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE, torch.float16)
        
        with torch.inference_mode():
            # max_new_tokens를 작게 설정하여 답변 생성 속도를 높이고 불필요한 텍스트 생성을 줄입니다.
            outputs = model.generate(
                **inputs,
                do_sample=False,
                num_beams=5,
                max_length=256,
                min_length=1,
                top_p=0.9,
                repetition_penalty=1.5,
                length_penalty=1.0,
                temperature=1,
            )
            generated_text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()

        # --- 생성된 텍스트에서 답변 파싱 ---
        # 생성된 텍스트에서 A, B, C, D 중 가장 먼저 나오는 알파벳을 찾습니다.
        # 모델이 보기의 내용을 그대로 생성할 수도 있으므로, 보기의 내용과도 비교합니다.
        prediction_label = 'N/A' # 기본값
        
        # 1. 알파벳 레이블 우선 검색
        match = re.search(r'\b([A-D])\b', generated_text)
        if match:
            prediction_label = match.group(1)
        else:
            # 2. 알파벳이 없으면 보기의 내용과 직접 비교
            for label, text in choices.items():
                if text.lower() in generated_text.lower():
                    prediction_label = label
                    break

        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = prediction_label

    except FileNotFoundError:
        print(f"\n경고: ID '{test_id}'의 이미지 파일을 찾을 수 없습니다.")
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = 'Error'
    except Exception as e:
        print(f"\n오류: ID '{test_id}' 처리 중 예외 발생: {e}")
        df_submission.loc[df_submission['ID'] == test_id, 'answer'] = 'Error'

# --- 5. 결과 저장 및 출력 ---
df_submission.to_csv(FINAL_SUBMISSION_PATH, index=False)
print("\n" + "="*50)
print("추론이 완료되었습니다.")
print(f"결과가 {FINAL_SUBMISSION_PATH} 파일에 저장되었습니다.")
print("\n최종 제출 파일 미리보기 (상위 5개):")
print(df_submission.head())
print("="*50)

1. 설정을 초기화합니다...
사용 디바이스: cuda

2. InstructBLIP 모델(Salesforce/instructblip-flan-t5-xl)을 로드합니다...


Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.57s/it]


모델 로드 완료.

3. 데이터 로드를 시작합니다...

4. 총 60개의 데이터에 대한 추론을 시작합니다...


InstructBLIP 추론 진행률:   2%|▏         | 1/60 [00:00<00:22,  2.57it/s]The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a 


오류: ID 'TEST_000' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_001' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_002' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_003' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_004' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_005' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!


...


오류: ID 'TEST_047' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_048' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_049' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_050' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_051' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!


The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
InstructBLIP 추론 진행률:  90%|█████████ | 54/60 [00:02<00:00, 25.25it/s]The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a


오류: ID 'TEST_052' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_053' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_054' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_055' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_056' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_057' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!


The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
InstructBLIP 추론 진행률: 100%|██████████| 60/60 [00:02<00:00, 22.78it/s]


오류: ID 'TEST_058' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

오류: ID 'TEST_059' 처리 중 예외 발생: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

추론이 완료되었습니다.
결과가 instructblip_submission.csv 파일에 저장되었습니다.

최종 제출 파일 미리보기 (상위 5개):
         ID answer
0  TEST_000  Error
1  TEST_001  Error
2  TEST_002  Error
3  TEST_003  Error
4  TEST_004  Error


In [3]:
# 전체 파라미터와 학습 가능한 파라미터 수를 계산
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}개 ({total_params / 1_000_000:.2f}M)")
print(f"학습 가능한 파라미터: {trainable_params:,}개 ({trainable_params / 1_000_000:.2f}M)")

전체 파라미터: 4,022,969,088개 (4022.97M)
학습 가능한 파라미터: 4,022,969,088개 (4022.97M)
